# Direct classification

In this notebook we attempt the following improvements to obtain a better accuracy score.

- Directly classifying health status from gut biome, rather than a two-step pipeline
- Accounting for the compositional nature of the microbiome data using a central log-ratio (CLR) transformation
- Over/under-sampling or weighting to account for class imbalance during training.

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, roc_curve
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from tqdm.auto import tqdm, trange

pd.set_option("display.precision", 3)
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


## 1. Data loading

Load the pre-processed AGP dataset: `meta.tsv` (id, age, health) and `otu.tsv` (~5,966 samples × 942 OTUs after 10% prevalence filtering — see `prepare_data.py`).

The paper's GAI pipeline trains a regression model on the **healthy** cohort, then derives a Gut Age Index for classification. Here we skip that indirection and classify **directly** from OTU abundances.

In [9]:
DATA = "data/processed/AGP"
meta = pd.read_csv(f"{DATA}/meta.tsv", sep="\t", index_col="id")
otu  = pd.read_csv(f"{DATA}/otu.tsv",  sep="\t", index_col="id")

# Align indices
common = meta.index.intersection(otu.index)
meta, otu = meta.loc[common], otu.loc[common]

y = (meta["health"] == "n").astype(int).values
print(f"Samples: {len(y)}  |  Healthy: {(y==0).sum()}  |  Non-healthy: {(y==1).sum()}  |  Prevalence: {y.mean():.1%}")
print(f"OTU features: {otu.shape[1]}")

Samples: 5965  |  Healthy: 1852  |  Non-healthy: 4113  |  Prevalence: 69.0%
OTU features: 1329


## 2. CLR transformation

Microbiome counts are **compositional** — relative abundances sum to a constant per sample, so standard Euclidean distances are misleading (Aitchison, 1986). The Centered Log-Ratio addresses this:

$$\text{CLR}(x_i) = \ln(x_i + 1) - \frac{1}{D}\sum_{j=1}^{D} \ln(x_j + 1)$$

The pseudocount of 1.0 handles the many zeros in sparse OTU data. After CLR, each sample's transformed values sum to zero — the data lives in a proper Euclidean subspace where standard ML methods apply.

In [10]:
def clr_transform(df, pseudocount=1.0):
    """Centered log-ratio transform. Rows = samples, cols = OTUs."""
    log_mat = np.log(df.values + pseudocount)
    return log_mat - log_mat.mean(axis=1, keepdims=True)

X = clr_transform(otu)
print(f"X shape: {X.shape}")
print(f"Row sums ≈ 0 check: max |row sum| = {np.abs(X.sum(axis=1)).max():.2e}")

X shape: (5965, 1329)
Row sums ≈ 0 check: max |row sum| = 1.02e-11


## 3. Model definitions

Three classifiers of increasing complexity, all handling the ~2:1 class imbalance via balanced weights:

| Model | Type | Rationale |
|---|---|---|
| **Elastic Net** | Linear (L1+L2) | Sparse baseline — automatic feature selection among 942 OTUs |
| **Random Forest** | Ensemble of trees | Nonlinear, robust to correlated & high-dimensional features |
| **MLP** | Neural network | Can learn complex OTU interaction patterns |

In [11]:
class MicrobiomeDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class BaselineMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 128),       nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 2),
        )
    def forward(self, x):
        return self.net(x)

## 4. Training helpers

The MLP training loop. Sklearn models (Elastic Net, Random Forest) need no custom loop — `fit` / `predict_proba` handle everything.

Class imbalance is addressed via `sklearn.utils.class_weight.compute_class_weight('balanced')`, passed to both `class_weight=` for sklearn models and `nn.CrossEntropyLoss(weight=...)` for the MLP.

In [12]:
def train_mlp(X_tr, y_tr, X_val, y_val, n_epochs=20, lr=1e-3, batch_size=256):
    """Train MLP for one fold, return (val_probs, val_preds)."""
    # Class weights
    cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_tr)
    weights = torch.tensor(cw, dtype=torch.float32).to(DEVICE)

    model = BaselineMLP(X_tr.shape[1]).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    train_ds = MicrobiomeDataset(X_tr, y_tr)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    for epoch in trange(n_epochs, desc="    MLP epochs", leave=False):
        model.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            criterion(model(xb), yb).backward()
            optimizer.step()

    # Validation
    model.eval()
    with torch.no_grad():
        xv = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)
        logits = model(xv)
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
    return probs

## 5. Cross-validated evaluation

5-fold stratified CV, shared across all models. We store out-of-fold probabilities to compute overall AUC and per-fold balanced accuracy.

In [ ]:
models = {
    "Elastic Net": lambda: LogisticRegression(
        penalty="elasticnet", solver="saga", l1_ratio=0.5,
        class_weight="balanced", max_iter=2000, random_state=SEED,
    ),
    "Random Forest": lambda: RandomForestClassifier(
        n_estimators=500, class_weight="balanced", n_jobs=-1, random_state=SEED,
    ),
    "MLP": None,  # handled separately
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
results = {name: {"ba": [], "auc": [], "oof_probs": np.zeros(len(y))} for name in models}

for fold, (tr_idx, val_idx) in enumerate(tqdm(list(cv.split(X, y)), desc="CV folds")):
    X_tr, X_val = X[tr_idx], X[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    for name, make_model in tqdm(models.items(), desc=f"  Fold {fold} models", leave=False):
        if name == "MLP":
            probs = train_mlp(X_tr, y_tr, X_val, y_val)
        else:
            clf = make_model()
            clf.fit(X_tr, y_tr)
            probs = clf.predict_proba(X_val)[:, 1]

        preds = (probs >= 0.5).astype(int)
        ba  = balanced_accuracy_score(y_val, preds)
        auc = roc_auc_score(y_val, probs)
        results[name]["ba"].append(ba)
        results[name]["auc"].append(auc)
        results[name]["oof_probs"][val_idx] = probs

    print(f"Fold {fold}: " + " | ".join(
        f"{n}: BA={results[n]['ba'][-1]:.3f} AUC={results[n]['auc'][-1]:.3f}" for n in models
    ))

CV folds:   0%|          | 0/5 [00:00<?, ?it/s]

  Fold 0 models:   0%|          | 0/3 [00:00<?, ?it/s]

## 6. Results

Paper baseline: **68% balanced accuracy** (GAI thresholding on the same AGP cohort). Our target: **BA > 70%, AUC > 0.75**.

In [ ]:
rows = []
for name, res in results.items():
    ba, auc = np.array(res["ba"]), np.array(res["auc"])
    rows.append({
        "Model": name,
        "Balanced Acc (mean±std)": f"{ba.mean():.3f} ± {ba.std():.3f}",
        "AUC-ROC (mean±std)":     f"{auc.mean():.3f} ± {auc.std():.3f}",
        "BA mean": ba.mean(), "AUC mean": auc.mean(),
    })
summary = pd.DataFrame(rows).sort_values("BA mean", ascending=False)
summary[["Model", "Balanced Acc (mean±std)", "AUC-ROC (mean±std)"]]

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
colors = {"Elastic Net": "#4e79a7", "Random Forest": "#59a14f", "MLP": "#e15759"}

for name, res in results.items():
    fpr, tpr, _ = roc_curve(y, res["oof_probs"])
    auc_val = roc_auc_score(y, res["oof_probs"])
    ax.plot(fpr, tpr, color=colors[name], label=f"{name} (AUC={auc_val:.3f})")

ax.plot([0, 1], [0, 1], "k--", linewidth=0.5)
ax.set(xlabel="False positive rate", ylabel="True positive rate", title="ROC — out-of-fold")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()